# Evaneos "Indonesia 360": destination analysis of trip itineraries

Data-analysis code I wrote during my internship at Wise Steps Consulting (2025) for **Evaneos**, a Paris-based travel marketplace, as part of the INDO360 sustainable-tourism study on overtourism and carbon emissions.

The real input was the client's booking export: about **29,000 raw records**, cleaned into **16,000+ analysis-ready itinerary-day rows** covering about **1,200 trips**. This published version contains **no client data and no results**: all cell outputs are removed, the column names are generic, and the notebook runs on a small fictional sample so the method can be reviewed end to end.

**What the notebook does**

1. Load an itinerary table (one row per trip day).
2. Tag every day with Indonesian destinations using a keyword dictionary and regex word boundaries over the free-text itinerary fields (multilingual, inconsistent spelling).
3. Lift the day-level tags to trip level, so each trip knows every destination it visits.
4. Compare **share of trips** with **share of real days** per destination (popularity vs time actually spent).
5. Cross-tabulate season against Bali vs non-Bali trips.

## 1. Load the data

Expected columns: `trip_id`, `day_index` (0 = first day of the trip), `day_title`, `accommodation`, `transport`, `activity`, `destination`, `season`.
Set `DATA_PATH` to your own export to run on real data; leave it as `None` to use the fictional sample.

In [ ]:
import pandas as pd
import re

DATA_PATH = None  # e.g. 'itineraries.xlsx'


def sample_itineraries() -> pd.DataFrame:
    """A tiny fictional dataset with the same shape as the real export."""
    rows = [
        # trip_id, day_index, day_title, accommodation, transport, activity, destination, season
        ('T001', 0, 'Arrival in Denpasar', 'Sanur Beach Homestay', 'Airport transfer', None, 'Indonesia', 'High'),
        ('T001', 1, 'Ubud rice terraces', 'Ubud Garden Villa', 'Private driver', 'Tegallalang walk', 'Indonesia', 'High'),
        ('T001', 2, 'Tirta Empul and Gunung Kawi', 'Ubud Garden Villa', 'Private driver', 'Temple visit', 'Indonesia', 'High'),
        ('T001', 3, 'Fast boat to Gili Air', 'Gili Air Bungalow', 'Fast boat', 'Snorkelling', 'Indonesia', 'High'),
        ('T002', 0, 'Arrival in Surabaya', 'City Hotel', 'Airport transfer', None, 'Java', 'Low'),
        ('T002', 1, 'Sunrise at Bromo', 'Bromo Lodge', 'Jeep', 'Volcano trek', 'Java', 'Low'),
        ('T002', 2, 'Ijen crater night hike', 'Banyuwangi Guesthouse', 'Private driver', 'Ijen hike', 'Java', 'Low'),
        ('T003', 0, 'Arrival in Medan', 'Medan Inn', 'Airport transfer', None, 'Sumatra', 'Shoulder'),
        ('T003', 1, 'Bukit Lawang jungle', 'Riverside Lodge', 'Private driver', 'Orangutan trek', 'Sumatra', 'Shoulder'),
        ('T003', 2, 'Onward to Bali', 'Jimbaran Bay Resort', 'Domestic flight', None, 'Sumatra', 'Shoulder'),
    ]
    columns = ['trip_id', 'day_index', 'day_title', 'accommodation', 'transport', 'activity', 'destination', 'season']
    return pd.DataFrame(rows, columns=columns)


df = pd.read_excel(DATA_PATH) if DATA_PATH else sample_itineraries()
display(df.head())

## 2. What the raw destination field says

The booking's own `destination` field is coarse (often just the country), which is why the analysis classifies the free-text itinerary instead.

In [ ]:
df_Fix = df
# Count the occurrences of each destination value
destination_counts_fix = df_Fix['destination'].value_counts()
display(destination_counts_fix)

## 3. Keyword-based destination tagging

Two flags per destination:

- **Daily flag** (`Bali`): 1 only when *this day's* text mentions the destination.
- **Trip flag** (`indicated Bali`): 1 on *every* day of a trip that visits the destination at least once, spread with `groupby(...).transform('max')`.

The keyword lists absorb real-world spelling variants (for example `Tegalalang`, `Tegallalang`, `Tegalallang`) and itinerary labels such as `denpasar-bali-ubud`.

In [ ]:
# Always start from fresh data
df_Fix = df.copy()

# Keyword dictionary: destination tag -> spellings seen in itinerary text
location_keywords = {
    'bali': ['bali', 'pura taman ayun', 'bali bird park', 'ubud', 'jimbaran', 'ubud-lombok', 'denpasar',
             'denpasar-bali-kerobokan', 'denpasar-bali', 'denpasar-bali-balangan', 'denpasar-bali-munduk',
             'bali-balangan', 'denpasar-bali-ubud', 'bali-ubud', 'nusa penida', 'sanur', 'bali-jimbaran',
             'bali-sanur', 'munduk', 'balinese', 'balinaise', 'kuta', 'uluwatu', 'denpasar-bali-canggu',
             'bali -ubud', 'menjangan', 'batur', 'kintamani', 'tabanan', 'candidasa', 'sidemen'],
    'canggu': ['Canggu'],
    'nusa_penida': ['Nusa Penida', 'nusapenida'],
    'tirta_empul': ['Tirta Empul', 'tirtaempul'],
    'ubud': ['Ubud'],
    'tegalalang': ['Tegalalang', 'Tegalallang', 'Tegallalang'],
    'java': ['Bromo', 'Ijen', 'Banyuwangi', 'Probolinggo', 'Surabaya', 'Ketapang', 'Gombengsari', 'Java', 'Jawa', 'Magelang', 'Solo'],
    'sumatra': ['Sumatra', 'Medan', 'Bukit Lawang', 'Bohorok', 'Harau', 'Orangutan', 'Brastagi', 'Padang', 'Orang Outangs', 'Tangkahan'],
    'lombok': ['Lombok', 'Gili Trawangan', 'Gili Meno', 'Gili Air', 'Rinjani', 'Praya'],
    'batur_volcano': ['Batur'],
    'jatiluwih': ['jatiluwih', 'Jatiluwih'],
    'lake_tamblingan': ['Tamblingan'],
    'gunung_kawi': ['Kawi', 'gunungkawi', 'Gunung Kawi'],
    'lake_bratan': ['Bratan', 'Beratan'],
    'besakih': ['Besakih'],
    'taman_sari': ['Taman Sari', 'Tamansari'],
    'penglipuran': ['Penglipuran', 'Panglipuran'],
    'kumulilir': ['Kumulilir']
}

# --- Step 1: build one lower-case search text per day ---
target_cols = ['day_title', 'accommodation', 'transport', 'activity']
existing_cols = [col for col in target_cols if col in df_Fix.columns]
df_Fix['search_text'] = df_Fix[existing_cols].fillna('').astype(str).agg(' '.join, axis=1).str.lower()

# --- Step 2: daily flag and trip flag per destination ---
print("Tagging destinations...")
all_indicated_cols = []

for tag_key, keywords in location_keywords.items():
    dest_name = tag_key.replace('_', ' ').title()   # e.g. 'Nusa Penida'
    raw_col = dest_name                              # daily flag
    ind_col = f'indicated {dest_name}'               # trip flag

    # A. One regex per destination, whole words only
    pattern = r'\b(?:' +'|'.join([re.escape(k.lower()) for k in keywords]) + r')\b'

    # B. Daily flag: does this day mention the destination?
    df_Fix[raw_col] = df_Fix['search_text'].str.contains(pattern, regex=True).astype(int)

    # C. Trip flag: spread a 1 to every day of the same trip
    df_Fix[ind_col] = df_Fix.groupby('trip_id')[raw_col].transform('max')

    all_indicated_cols.append(ind_col)

df_Fix.drop(columns=['search_text'], inplace=True)

# --- Step 3: check on day 0 (one row per trip) ---
print("Done. Compare the daily flag 'Bali' with the trip flag 'indicated Bali':")
df_day_0 = df_Fix[df_Fix['day_index'] == 0]
cols_to_show = ['trip_id', 'day_index', 'Bali', 'indicated Bali', 'Lombok', 'indicated Lombok']
display(df_day_0[df_day_0['indicated Bali'] == 1][cols_to_show].head(5))

## 4. Share of trips vs share of real days

- **Trips**: how many trips visit a destination (trip flag on day 0, so each trip counts once).
- **Real days**: how many itinerary days are actually spent there (daily flag over all rows).

A destination that appears in many trips but holds few days is a pass-through stop; one that holds many days concentrates visitor pressure, which fed the overtourism workstream.

In [ ]:
df_day_0 = df_Fix[df_Fix['day_index'] == 0]

total_unique_trips = len(df_day_0)  # one day-0 row per trip
total_real_rows = len(df_Fix)       # all itinerary days

final_analysis = {}

if total_real_rows == 0:
    print("No rows to analyse.")
else:
    for tag_key in location_keywords.keys():
        dest_name = tag_key.replace("_", " ").title()
        col_daily = dest_name
        col_indicated = f'indicated {dest_name}'

        if col_daily in df_Fix.columns and col_indicated in df_Fix.columns:
            # A. Trips that visit the destination
            count_trips = df_day_0[col_indicated].sum()
            pct_trips = (count_trips / total_unique_trips) * 100

            # B. Real days spent there (daily flag, not the trip flag)
            count_real_days = df_Fix[col_daily].sum()
            pct_real_days = (count_real_days / total_real_rows) * 100

            final_analysis[dest_name] = {
                'Trips': int(count_trips),
                '% Trips': f'{pct_trips:.2f}%',
                'Real days': int(count_real_days),
                '% Real days': f'{pct_real_days:.2f}%'
            }
        else:
            print(f"Warning: columns for '{dest_name}' are missing.")

    print(f"Unique trips (day 0): {total_unique_trips}")
    print(f"Itinerary days (all rows): {total_real_rows}")

    if final_analysis:
        results_df = pd.DataFrame.from_dict(final_analysis, orient='index')
        display(results_df.sort_values(by='Real days', ascending=False))

## 5. Seasonality: Bali vs non-Bali trips

In [ ]:
df_day_0 = df_Fix[df_Fix['day_index'] == 0]

season_bali_analysis = pd.crosstab(
    index=df_day_0['season'],
    columns=df_day_0['indicated Bali']
)

# Columns 0 and 1 may not both exist (e.g. every trip touches Bali)
rename_map = {0: 'Non-Bali (Other)', 1: 'Bali Trip'}
season_bali_analysis = season_bali_analysis.rename(columns=rename_map)
season_bali_analysis['Total Request'] = season_bali_analysis.sum(axis=1)

if 'Bali Trip' in season_bali_analysis.columns:
    season_bali_analysis['% Bali'] = (season_bali_analysis['Bali Trip'] / season_bali_analysis['Total Request']) * 100
    season_bali_analysis['% Bali'] = season_bali_analysis['% Bali'].map('{:.1f}%'.format)

print("--- Season: Bali vs non-Bali trips (day 0, one row per trip) ---")
display(season_bali_analysis.sort_values(by='Total Request', ascending=False))

## 6. Export (optional)

```python
df_Fix.to_excel('processed_itineraries.xlsx', index=False)
```